# Large Corpus Downloader Notebook

> Hands-on Build It and Exercises.

## Build It

`code/main.py` implements:

- `ShardPlanner` - reads a list of shard URLs and produces planned manifest entries.

- `StreamingDownloader` - opens a `urllib` stream with optional `Range`, writes to a temporary file, updates the `.partial.json` checkpoint on every chunk, and verifies the sha256 prefix on resume.

- `ZstdDocIterator` - wraps the file-like stream in `zstandard.ZstdDecompressor` and yields one document per line.

- `MinHasher` - produces a `k`-component signature for a string using a fixed family of hash seeds.

- `LSHIndex` - buckets signatures by band and reports collisions.

- `Dedup` - combines hasher and index to label each document `keep` or `near_duplicate` along with the matching shard id.

- `ManifestWriter` - collects per-shard stats and writes `manifest.json`.

A demo at the bottom of the file builds a small synthetic corpus on disk, compresses it with `zstandard`, downloads it through a `file://` URL, deduplicates, and prints the manifest.

Run it:

In [ ]:
```bash

python3 code/main.py

In [ ]:
```

The script exits zero and prints a manifest summary.

## Exercises

In [ ]:
1. Add a `--shingle-width` flag and measure how the dedup verdict changes at widths 3, 5, 9. Defend the chosen default.
2. Add gzip support next to zstd by sniffing the magic bytes. The downloader should not require the caller to specify the codec.
3. Add a `--resume-only` mode that refuses to start a fresh download if no checkpoint is found. Useful in CI to keep one run from accidentally re-pulling 200 GB.
4. Move the LSH index to a shelf or sqlite file and measure throughput vs the in-memory variant.
5. Add a manifest sha256 check on startup. The downloader should fail closed if the manifest on disk disagrees with the manifest hash in `manifest.lock`.